# Simulate Feedback

Development notebook to simulate implicit user behaviour and inspect how it modifies the Bayesian Network CPDs.

Workflow:
1. Load the BN model and initialise CPT counts
2. Run `simulate_session` calls (individual or batch)
3. Inspect CPD diffs and probability tables after each session

## Setup

In [1]:
import sys
import os
import copy
import itertools
import pandas as pd

# Make sure main/ is on the path when running from repo root
MODULE_DIR = os.path.dirname(os.path.abspath('__file__'))
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

import feedback as fb
import graph_builder
from simulate_feedback import (
    viewing_to_feedback,
    simulate_session,
    snapshot_counts,
    _counts_to_probs,
    print_cpd_diff,
)

In [2]:
MODEL_PATH = os.path.join(MODULE_DIR, 'output', 'model.pkl')

print('Loading model ...')
model = graph_builder.load_model(MODEL_PATH)

print('Initialising CPT counts ...')
cpt_counts = fb.initialize_cpt_counts(model)

print('Done. Variables:', list(cpt_counts.keys()))

Loading model ...
Initialising CPT counts ...
Done. Variables: ['UserAge', 'ProgramType', 'ProgramGenre', 'ProgramDuration', 'UserGender', 'HouseholdType', 'TimeOfDay', 'DayType']


## Helper: visualise a full CPD as a DataFrame

In [3]:
def cpd_to_df(cpt_counts, variable):
    """Return a tidy DataFrame with the current probability distribution for *variable*."""
    info = cpt_counts[variable]
    probs = _counts_to_probs(info, variable)
    parents = info['parents']
    var_states = info['state_names'][variable]

    rows = []
    for parent_state, dist in probs.items():
        row = {p: s for p, s in zip(parents, parent_state)} if parents else {}
        row.update(dist)
        rows.append(row)

    df = pd.DataFrame(rows)
    if parents:
        df = df.set_index(parents)
    return df.round(4)


def cpd_diff_df(cpt_before, cpt_after, variable):
    """Return a DataFrame showing the delta (after - before) for each cell."""
    df_before = cpd_to_df(cpt_before, variable)
    df_after  = cpd_to_df(cpt_after,  variable)
    return (df_after - df_before).round(6)

## Viewing → feedback rule explorer

Quickly preview what feedback label and learning rate a given viewing event would produce.

In [4]:
cases = [
    (100,  1, None),   # full watch, single
    (100,  3, None),   # full watch, rewatch x3
    ( 80,  1, None),   # >70% → accepted
    ( 60,  1, None),   # <70% → rejected
    ( 30,  1, None),   # low %
    ( 80,  1, 4),      # 80% of 4 min = 3.2 min → short-watch guard
    ( 20,  1, 90),     # 20% of 90 min = 18 min → % rule applies
]

rows = []
for pct, times, dur in cases:
    result = viewing_to_feedback(pct, times, dur)
    rows.append({
        'percent_watched': pct,
        'times_watched': times,
        'duration_min': dur,
        'user_feedback': result['user_feedback'],
        'learning_rate': result['learning_rate'],
    })

pd.DataFrame(rows)

,percent_watched,times_watched,duration_min,user_feedback,learning_rate
0,100,1,NaN,accepted,50
1,100,3,NaN,accepted,150
2,80,1,NaN,accepted,25
3,60,1,NaN,rejected,70
4,30,1,NaN,rejected,70
5,80,1,4.0,rejected,100
6,20,1,90.0,rejected,70


## Example 1 – Strong acceptance: watched 100%, senior couple, weekday afternoon

In [5]:
context_couple_afternoon = {
    'DayType':       'weekday',
    'TimeOfDay':     'afternoon',
    'HouseholdType': 'couple',
    'UserAge':       'senior',
    'UserGender':    'female',
}

before_1 = snapshot_counts(cpt_counts)

simulate_session(
    model, cpt_counts,
    program_type='series',
    program_genre='drama',
    percent_watched=100,
    context_attrs=context_couple_afternoon,
)

[simulate] series/drama @ 100% (x1) -> accepted, lr=50

 Applying feedback: accepted for Type=series, Genre=drama


{'user_feedback': 'accepted', 'learning_rate': 50}

In [6]:
print('--- ProgramType delta ---')
display(cpd_diff_df(before_1, cpt_counts, 'ProgramType'))

print('--- ProgramGenre delta ---')
display(cpd_diff_df(before_1, cpt_counts, 'ProgramGenre'))

--- ProgramType delta ---


documentary  \
DayType HouseholdType TimeOfDay UserAge UserGender                
weekday couple        afternoon adult   female           0.0000   
                                        male             0.0000   
                                senior  female          -0.0832   
                                        male             0.0000   
                                young   female           0.0000   
...                                                         ...   
weekend single        night     adult   male             0.0000   
                                senior  female           0.0000   
                                        male             0.0000   
                                young   female           0.0000   
                                        male             0.0000   

                                                    entertainment   movie  \
DayType HouseholdType TimeOfDay UserAge UserGender                          
weekday couple        afternoon adult   female             0.0000  0.0000   
                                        male               0.0000  0.0000   
                                senior  female            -0.0005 -0.0575   
                                        male               0.0000  0.0000   
                                young   female             0.0000  0.0000   
...                                                           ...     ...   
weekend single        night     adult   male               0.0000  0.0000   
                                senior  female             0.0000  0.0000   
                                        male               0.0000  0.0000   
                                young   female             0.0000  0.0000   
                                        male               0.0000  0.0000   

                                                      news  series  
DayType HouseholdType TimeOfDay UserAge UserGender                  
weekday couple        afternoon adult   female      0.0000  0.0000  
                                        male        0.0000  0.0000  
                                senior  female     -0.1915  0.3328  
                                        male        0.0000  0.0000  
                                young   female      0.0000  0.0000  
...                                                    ...     ...  
weekend single        night     adult   male        0.0000  0.0000  
                                senior  female      0.0000  0.0000  
                                        male        0.0000  0.0000  
                                young   female      0.0000  0.0000  
                                        male        0.0000  0.0000  

[108 rows x 5 columns]

--- ProgramGenre delta ---


comedy  documentary   drama  entertainment  \
DayType ProgramType   UserGender                                               
weekday documentary   female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        entertainment female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        movie         female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        news          female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        series        female     -0.1578      -0.0002  0.1936        -0.0002   
                      male        0.0000       0.0000  0.0000         0.0000   
weekend documentary   female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        entertainment female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        movie         female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        news          female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        series        female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   

                                  horror    news  romance  
DayType ProgramType   UserGender                           
weekday documentary   female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        entertainment female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        movie         female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        news          female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        series        female     -0.0349 -0.0002  -0.0002  
                      male        0.0000  0.0000   0.0000  
weekend documentary   female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        entertainment female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        movie         female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        news          female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        series        female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000

## Example 2 – Rejected: watched only 60% (below 70% threshold)

In [7]:
before_2 = snapshot_counts(cpt_counts)

simulate_session(
    model, cpt_counts,
    program_type='movie',
    program_genre='comedy',
    percent_watched=60,
)

[simulate] movie/comedy @ 60% (x1) -> rejected, lr=70

 Applying feedback: rejected for Type=movie, Genre=comedy


{'user_feedback': 'rejected', 'learning_rate': 70}

In [8]:
print('--- ProgramType delta (expect rejection signal) ---')
display(cpd_diff_df(before_2, cpt_counts, 'ProgramType'))

print('--- ProgramGenre delta ---')
display(cpd_diff_df(before_2, cpt_counts, 'ProgramGenre'))

--- ProgramType delta (expect rejection signal) ---


documentary  \
DayType HouseholdType TimeOfDay UserAge UserGender                
weekday couple        afternoon adult   female           0.0001   
                                        male             0.0001   
                                senior  female           0.0005   
                                        male             0.0008   
                                young   female           0.0000   
...                                                         ...   
weekend single        night     adult   male             0.0008   
                                senior  female           0.0008   
                                        male             0.0009   
                                young   female           0.0008   
                                        male             0.0008   

                                                    entertainment   movie  \
DayType HouseholdType TimeOfDay UserAge UserGender                          
weekday couple        afternoon adult   female            -0.0035  0.0091   
                                        male              -0.0036  0.0091   
                                senior  female             0.0005 -0.0022   
                                        male               0.0008 -0.0033   
                                young   female            -0.0034  0.0091   
...                                                           ...     ...   
weekend single        night     adult   male               0.0008 -0.0033   
                                senior  female             0.0008 -0.0033   
                                        male               0.0008 -0.0032   
                                young   female             0.0008 -0.0032   
                                        male               0.0008 -0.0033   

                                                      news  series  
DayType HouseholdType TimeOfDay UserAge UserGender                  
weekday couple        afternoon adult   female     -0.0020 -0.0035  
                                        male       -0.0035 -0.0019  
                                senior  female      0.0005  0.0005  
                                        male        0.0008  0.0008  
                                young   female     -0.0023 -0.0033  
...                                                    ...     ...  
weekend single        night     adult   male        0.0008  0.0008  
                                senior  female      0.0008  0.0008  
                                        male        0.0008  0.0008  
                                young   female      0.0008  0.0008  
                                        male        0.0008  0.0008  

[108 rows x 5 columns]

--- ProgramGenre delta ---


comedy  documentary   drama  entertainment  \
DayType ProgramType   UserGender                                               
weekday documentary   female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        entertainment female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        movie         female     -0.0552       0.0092  0.0092         0.0092   
                      male       -0.0482       0.0081  0.0080         0.0081   
        news          female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        series        female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
weekend documentary   female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        entertainment female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        movie         female     -0.0477       0.0079  0.0079         0.0079   
                      male       -0.0612       0.0102  0.0102         0.0102   
        news          female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        series        female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   

                                  horror    news  romance  
DayType ProgramType   UserGender                           
weekday documentary   female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        entertainment female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        movie         female      0.0092  0.0092   0.0092  
                      male        0.0081  0.0081   0.0081  
        news          female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        series        female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
weekend documentary   female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        entertainment female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        movie         female      0.0080  0.0079   0.0079  
                      male        0.0102  0.0102   0.0102  
        news          female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        series        female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000

## Example 3 – Rejection: watched 30%, weekend morning

In [9]:
before_3 = snapshot_counts(cpt_counts)

simulate_session(
    model, cpt_counts,
    program_type='news',
    program_genre='news',
    percent_watched=30,
    context_attrs={'DayType': 'weekend', 'TimeOfDay': 'morning'},
)

[simulate] news/news @ 30% (x1) -> rejected, lr=70

 Applying feedback: rejected for Type=news, Genre=news


{'user_feedback': 'rejected', 'learning_rate': 70}

In [10]:
print('--- ProgramType delta ---')
display(cpd_diff_df(before_3, cpt_counts, 'ProgramType'))

print('--- ProgramGenre delta ---')
display(cpd_diff_df(before_3, cpt_counts, 'ProgramGenre'))

--- ProgramType delta ---


documentary  \
DayType HouseholdType TimeOfDay UserAge UserGender                
weekday couple        afternoon adult   female              0.0   
                                        male                0.0   
                                senior  female              0.0   
                                        male                0.0   
                                young   female              0.0   
...                                                         ...   
weekend single        night     adult   male                0.0   
                                senior  female              0.0   
                                        male                0.0   
                                young   female              0.0   
                                        male                0.0   

                                                    entertainment  movie  \
DayType HouseholdType TimeOfDay UserAge UserGender                         
weekday couple        afternoon adult   female                0.0    0.0   
                                        male                  0.0    0.0   
                                senior  female                0.0    0.0   
                                        male                  0.0    0.0   
                                young   female                0.0    0.0   
...                                                           ...    ...   
weekend single        night     adult   male                  0.0    0.0   
                                senior  female                0.0    0.0   
                                        male                  0.0    0.0   
                                young   female                0.0    0.0   
                                        male                  0.0    0.0   

                                                    news  series  
DayType HouseholdType TimeOfDay UserAge UserGender                
weekday couple        afternoon adult   female       0.0     0.0  
                                        male         0.0     0.0  
                                senior  female       0.0     0.0  
                                        male         0.0     0.0  
                                young   female       0.0     0.0  
...                                                  ...     ...  
weekend single        night     adult   male         0.0     0.0  
                                senior  female       0.0     0.0  
                                        male         0.0     0.0  
                                young   female       0.0     0.0  
                                        male         0.0     0.0  

[108 rows x 5 columns]

--- ProgramGenre delta ---


comedy  documentary   drama  entertainment  \
DayType ProgramType   UserGender                                               
weekday documentary   female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        entertainment female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        movie         female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        news          female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        series        female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
weekend documentary   female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        entertainment female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        movie         female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        news          female      0.0291       0.0291  0.0291         0.0291   
                      male        0.0292       0.0292  0.0292         0.0292   
        series        female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   

                                  horror   news  romance  
DayType ProgramType   UserGender                          
weekday documentary   female      0.0000  0.000   0.0000  
                      male        0.0000  0.000   0.0000  
        entertainment female      0.0000  0.000   0.0000  
                      male        0.0000  0.000   0.0000  
        movie         female      0.0000  0.000   0.0000  
                      male        0.0000  0.000   0.0000  
        news          female      0.0000  0.000   0.0000  
                      male        0.0000  0.000   0.0000  
        series        female      0.0000  0.000   0.0000  
                      male        0.0000  0.000   0.0000  
weekend documentary   female      0.0000  0.000   0.0000  
                      male        0.0000  0.000   0.0000  
        entertainment female      0.0000  0.000   0.0000  
                      male        0.0000  0.000   0.0000  
        movie         female      0.0000  0.000   0.0000  
                      male        0.0000  0.000   0.0000  
        news          female      0.0291 -0.175   0.0291  
                      male        0.0292 -0.175   0.0292  
        series        female      0.0000  0.000   0.0000  
                      male        0.0000  0.000   0.0000

## Example 4 – Short-watch guard: 10% of 90 min → strong rejection

In [11]:
before_4 = snapshot_counts(cpt_counts)

simulate_session(
    model, cpt_counts,
    program_type='documentary',
    program_genre='documentary',
    percent_watched=10,
    duration_minutes=90,
)

[simulate] documentary/documentary @ 10% (x1) -> rejected, lr=100

 Applying feedback: rejected for Type=documentary, Genre=documentary


{'user_feedback': 'rejected', 'learning_rate': 100}

In [12]:
print('--- ProgramType delta ---')
display(cpd_diff_df(before_4, cpt_counts, 'ProgramType'))

print('--- ProgramGenre delta ---')
display(cpd_diff_df(before_4, cpt_counts, 'ProgramGenre'))

--- ProgramType delta ---


documentary  \
DayType HouseholdType TimeOfDay UserAge UserGender                
weekday couple        afternoon adult   female           0.0089   
                                        male             0.0089   
                                senior  female          -0.0030   
                                        male            -0.0047   
                                young   female           0.0090   
...                                                         ...   
weekend single        night     adult   male             0.0056   
                                senior  female          -0.0046   
                                        male            -0.0047   
                                young   female           0.0057   
                                        male             0.0058   

                                                    entertainment   movie  \
DayType HouseholdType TimeOfDay UserAge UserGender                          
weekday couple        afternoon adult   female            -0.0034  0.0000   
                                        male              -0.0036  0.0000   
                                senior  female             0.0008  0.0008   
                                        male               0.0012  0.0012   
                                young   female            -0.0034  0.0000   
...                                                           ...     ...   
weekend single        night     adult   male              -0.0011 -0.0022   
                                senior  female             0.0012  0.0012   
                                        male               0.0012  0.0011   
                                young   female            -0.0014 -0.0027   
                                        male              -0.0010 -0.0032   

                                                      news  series  
DayType HouseholdType TimeOfDay UserAge UserGender                  
weekday couple        afternoon adult   female     -0.0021 -0.0034  
                                        male       -0.0035 -0.0019  
                                senior  female      0.0008  0.0008  
                                        male        0.0012  0.0012  
                                young   female     -0.0023 -0.0033  
...                                                    ...     ...  
weekend single        night     adult   male        0.0002 -0.0024  
                                senior  female      0.0011  0.0012  
                                        male        0.0012  0.0012  
                                young   female      0.0002 -0.0019  
                                        male        0.0001 -0.0018  

[108 rows x 5 columns]

--- ProgramGenre delta ---


comedy  documentary   drama  entertainment  \
DayType ProgramType   UserGender                                               
weekday documentary   female      0.0208       -0.125  0.0208         0.0208   
                      male        0.0208       -0.125  0.0208         0.0208   
        entertainment female      0.0000        0.000  0.0000         0.0000   
                      male        0.0000        0.000  0.0000         0.0000   
        movie         female      0.0000        0.000  0.0000         0.0000   
                      male        0.0000        0.000  0.0000         0.0000   
        news          female      0.0000        0.000  0.0000         0.0000   
                      male        0.0000        0.000  0.0000         0.0000   
        series        female      0.0000        0.000  0.0000         0.0000   
                      male        0.0000        0.000  0.0000         0.0000   
weekend documentary   female      0.0208       -0.125  0.0208         0.0208   
                      male        0.0209       -0.125  0.0209         0.0209   
        entertainment female      0.0000        0.000  0.0000         0.0000   
                      male        0.0000        0.000  0.0000         0.0000   
        movie         female      0.0000        0.000  0.0000         0.0000   
                      male        0.0000        0.000  0.0000         0.0000   
        news          female      0.0000        0.000  0.0000         0.0000   
                      male        0.0000        0.000  0.0000         0.0000   
        series        female      0.0000        0.000  0.0000         0.0000   
                      male        0.0000        0.000  0.0000         0.0000   

                                  horror    news  romance  
DayType ProgramType   UserGender                           
weekday documentary   female      0.0208  0.0208   0.0208  
                      male        0.0208  0.0208   0.0208  
        entertainment female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        movie         female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        news          female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        series        female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
weekend documentary   female      0.0208  0.0208   0.0208  
                      male        0.0209  0.0209   0.0209  
        entertainment female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        movie         female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        news          female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000  
        series        female      0.0000  0.0000   0.0000  
                      male        0.0000  0.0000   0.0000

## Example 5 – Rewatch x3: learning_rate tripled (strong positive signal)

In [13]:
before_5 = snapshot_counts(cpt_counts)

simulate_session(
    model, cpt_counts,
    program_type='movie',
    program_genre='romance',
    percent_watched=100,
    times_watched=3,
    context_attrs=context_couple_afternoon,
)

[simulate] movie/romance @ 100% (x3) -> accepted, lr=150

 Applying feedback: accepted for Type=movie, Genre=romance


{'user_feedback': 'accepted', 'learning_rate': 150}

In [14]:
print('--- ProgramType delta ---')
display(cpd_diff_df(before_5, cpt_counts, 'ProgramType'))

print('--- ProgramGenre delta ---')
display(cpd_diff_df(before_5, cpt_counts, 'ProgramGenre'))

--- ProgramType delta ---


documentary  \
DayType HouseholdType TimeOfDay UserAge UserGender                
weekday couple        afternoon adult   female            0.000   
                                        male              0.000   
                                senior  female           -0.082   
                                        male              0.000   
                                young   female            0.000   
...                                                         ...   
weekend single        night     adult   male              0.000   
                                senior  female            0.000   
                                        male              0.000   
                                young   female            0.000   
                                        male              0.000   

                                                    entertainment   movie  \
DayType HouseholdType TimeOfDay UserAge UserGender                          
weekday couple        afternoon adult   female             0.0000  0.0000   
                                        male               0.0000  0.0000   
                                senior  female            -0.0012  0.4431   
                                        male               0.0000  0.0000   
                                young   female             0.0000  0.0000   
...                                                           ...     ...   
weekend single        night     adult   male               0.0000  0.0000   
                                senior  female             0.0000  0.0000   
                                        male               0.0000  0.0000   
                                young   female             0.0000  0.0000   
                                        male               0.0000  0.0000   

                                                      news  series  
DayType HouseholdType TimeOfDay UserAge UserGender                  
weekday couple        afternoon adult   female      0.0000  0.0000  
                                        male        0.0000  0.0000  
                                senior  female     -0.1922 -0.1678  
                                        male        0.0000  0.0000  
                                young   female      0.0000  0.0000  
...                                                    ...     ...  
weekend single        night     adult   male        0.0000  0.0000  
                                senior  female      0.0000  0.0000  
                                        male        0.0000  0.0000  
                                young   female      0.0000  0.0000  
                                        male        0.0000  0.0000  

[108 rows x 5 columns]

--- ProgramGenre delta ---


comedy  documentary   drama  entertainment  \
DayType ProgramType   UserGender                                               
weekday documentary   female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        entertainment female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        movie         female     -0.1325      -0.0067 -0.2604        -0.0067   
                      male        0.0000       0.0000  0.0000         0.0000   
        news          female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        series        female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
weekend documentary   female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        entertainment female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        movie         female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        news          female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   
        series        female      0.0000       0.0000  0.0000         0.0000   
                      male        0.0000       0.0000  0.0000         0.0000   

                                  horror    news  romance  
DayType ProgramType   UserGender                           
weekday documentary   female      0.0000  0.0000    0.000  
                      male        0.0000  0.0000    0.000  
        entertainment female      0.0000  0.0000    0.000  
                      male        0.0000  0.0000    0.000  
        movie         female     -0.0301 -0.0067    0.443  
                      male        0.0000  0.0000    0.000  
        news          female      0.0000  0.0000    0.000  
                      male        0.0000  0.0000    0.000  
        series        female      0.0000  0.0000    0.000  
                      male        0.0000  0.0000    0.000  
weekend documentary   female      0.0000  0.0000    0.000  
                      male        0.0000  0.0000    0.000  
        entertainment female      0.0000  0.0000    0.000  
                      male        0.0000  0.0000    0.000  
        movie         female      0.0000  0.0000    0.000  
                      male        0.0000  0.0000    0.000  
        news          female      0.0000  0.0000    0.000  
                      male        0.0000  0.0000    0.000  
        series        female      0.0000  0.0000    0.000  
                      male        0.0000  0.0000    0.000

## Sandbox – custom session

Edit the parameters below and re-run to test any combination.

In [ ]:
# --- edit these ---
PROGRAM_TYPE    = 'series'
PROGRAM_GENRE   = 'thriller'
PERCENT_WATCHED = 85
TIMES_WATCHED   = 1
DURATION_MIN    = None
CONTEXT = {
    'DayType':   'weekday',
    'TimeOfDay': 'night',
    # 'UserAge':       'adult',
    # 'UserGender':    'male',
    # 'HouseholdType': 'single',
}
# ------------------

before_custom = snapshot_counts(cpt_counts)

simulate_session(
    model, cpt_counts,
    program_type=PROGRAM_TYPE,
    program_genre=PROGRAM_GENRE,
    percent_watched=PERCENT_WATCHED,
    times_watched=TIMES_WATCHED,
    duration_minutes=DURATION_MIN,
    context_attrs=CONTEXT,
)

[simulate] series/thriller @ 85% (x1) -> accepted, lr=25

 Applying feedback: accepted for Type=series, Genre=thriller


{'user_feedback': 'accepted', 'learning_rate': 25}

In [16]:
for var in ['ProgramType', 'ProgramGenre']:
    delta = cpd_diff_df(before_custom, cpt_counts, var)
    changed = delta[(delta.abs() > 1e-8).any(axis=1)]
    print(f'--- {var} delta (changed rows only) ---')
    display(changed if not changed.empty else '(no changes)')

print('\n--- Full ProgramType CPD (current state) ---')
display(cpd_to_df(cpt_counts, 'ProgramType'))

print('\n--- Full ProgramGenre CPD (current state) ---')
display(cpd_to_df(cpt_counts, 'ProgramGenre'))

--- ProgramType delta (changed rows only) ---


documentary  \
DayType HouseholdType TimeOfDay UserAge UserGender                
weekday couple        night     adult   female          -0.0001   
                                        male            -0.0001   
                                senior  female          -0.0033   
                                        male            -0.0037   
                                young   female          -0.0001   
                                        male            -0.0001   
        family        night     adult   female          -0.0001   
                                        male            -0.0001   
                                senior  female          -0.0032   
                                        male            -0.0031   
                                young   female          -0.0001   
                                        male            -0.0001   
        single        night     adult   female          -0.0001   
                                        male            -0.0001   
                                senior  female          -0.0033   
                                        male            -0.0031   
                                young   female          -0.0001   
                                        male            -0.0001   

                                                    entertainment   movie  \
DayType HouseholdType TimeOfDay UserAge UserGender                          
weekday couple        night     adult   female            -0.0024 -0.0047   
                                        male              -0.0027 -0.0065   
                                senior  female             0.0000 -0.0012   
                                        male               0.0000 -0.0022   
                                young   female            -0.0021 -0.0052   
                                        male              -0.0022 -0.0080   
        family        night     adult   female            -0.0046 -0.0033   
                                        male              -0.0047 -0.0023   
                                senior  female             0.0000 -0.0017   
                                        male              -0.0001 -0.0018   
                                young   female            -0.0059 -0.0034   
                                        male              -0.0039 -0.0037   
        single        night     adult   female            -0.0019 -0.0052   
                                        male              -0.0012 -0.0086   
                                senior  female             0.0000 -0.0020   
                                        male              -0.0001 -0.0024   
                                young   female            -0.0021 -0.0050   
                                        male              -0.0019 -0.0075   

                                                      news  series  
DayType HouseholdType TimeOfDay UserAge UserGender                  
weekday couple        night     adult   female     -0.0001  0.0073  
                                        male        0.0000  0.0093  
                                senior  female     -0.0090  0.0137  
                                        male       -0.0077  0.0137  
                                young   female     -0.0001  0.0075  
                                        male        0.0000  0.0103  
        family        night     adult   female      0.0000  0.0081  
                                        male        0.0000  0.0073  
                                senior  female     -0.0086  0.0137  
                                        male       -0.0086  0.0136  
                                young   female     -0.0001  0.0095  
                                        male        0.0000  0.0076  
        single        night     adult   female      0.0000  0.0072  
                                        male        0.0000  0.0100  
                                senior  female     -0.0083  0.0137

--- ProgramGenre delta (changed rows only) ---


comedy  documentary   drama  entertainment  \
DayType ProgramType UserGender                                               
weekday series      female     -0.0243       0.0000 -0.0471         0.0000   
                    male       -0.0472      -0.0001 -0.0415        -0.0001   

                                horror    news  romance  
DayType ProgramType UserGender                           
weekday series      female     -0.0054  0.0000   0.0000  
                    male       -0.0220 -0.0001  -0.0001


--- Full ProgramType CPD (current state) ---


documentary  \
DayType HouseholdType TimeOfDay UserAge UserGender                
weekday couple        afternoon adult   female           0.0098   
                                        male             0.0098   
                                senior  female           0.0819   
                                        male             0.1488   
                                young   female           0.0098   
...                                                         ...   
weekend single        night     adult   male             0.0099   
                                senior  female           0.1240   
                                        male             0.3465   
                                young   female           0.0099   
                                        male             0.0099   

                                                    entertainment   movie  \
DayType HouseholdType TimeOfDay UserAge UserGender                          
weekday couple        afternoon adult   female             0.3776  0.0099   
                                        male               0.3910  0.0099   
                                senior  female             0.0012  0.5568   
                                        male               0.0040  0.1506   
                                young   female             0.3712  0.0099   
...                                                           ...     ...   
weekend single        night     adult   male               0.2110  0.3759   
                                senior  female             0.0094  0.3263   
                                        male               0.0078  0.3169   
                                young   female             0.2395  0.4344   
                                        male               0.1787  0.5062   

                                                      news  series  
DayType HouseholdType TimeOfDay UserAge UserGender                  
weekday couple        afternoon adult   female      0.2251  0.3776  
                                        male        0.3821  0.2072  
                                senior  female      0.1922  0.1679  
                                        male        0.6927  0.0040  
                                young   female      0.2547  0.3545  
...                                                    ...     ...  
weekend single        night     adult   male        0.0045  0.3987  
                                senior  female      0.5309  0.0094  
                                        male        0.3210  0.0078  
                                young   female      0.0044  0.3118  
                                        male        0.0042  0.3009  

[108 rows x 5 columns]


--- Full ProgramGenre CPD (current state) ---


comedy  documentary   drama  entertainment  \
DayType ProgramType   UserGender                                               
weekday documentary   female      0.0244       0.8536  0.0244         0.0244   
                      male        0.0252       0.8489  0.0252         0.0252   
        entertainment female      0.0007       0.0007  0.0007         0.9957   
                      male        0.0007       0.0007  0.0007         0.9957   
        movie         female      0.0883       0.0044  0.1736         0.0044   
                      male        0.1926       0.0095  0.3685         0.0095   
        news          female      0.0008       0.0008  0.0008         0.0008   
                      male        0.0007       0.0007  0.0007         0.0007   
        series        female      0.2914       0.0004  0.5657         0.0004   
                      male        0.3779       0.0008  0.3315         0.0008   
weekend documentary   female      0.0327       0.8036  0.0327         0.0327   
                      male        0.0298       0.8214  0.0298         0.0298   
        entertainment female      0.3410       0.0015  0.0015         0.6516   
                      male        0.3034       0.0014  0.0014         0.6894   
        movie         female      0.1906       0.0109  0.4773         0.0109   
                      male        0.2448       0.0130  0.3358         0.0130   
        news          female      0.0317       0.0317  0.0317         0.0317   
                      male        0.0319       0.0319  0.0319         0.0319   
        series        female      0.4802       0.0016  0.4215         0.0016   
                      male        0.4334       0.0017  0.3758         0.0017   

                                  horror    news  romance  
DayType ProgramType   UserGender                           
weekday documentary   female      0.0244  0.0244   0.0244  
                      male        0.0252  0.0252   0.0252  
        entertainment female      0.0007  0.0007   0.0007  
                      male        0.0007  0.0007   0.0007  
        movie         female      0.0201  0.0044   0.7047  
                      male        0.2306  0.0095   0.1799  
        news          female      0.0008  0.9952   0.0008  
                      male        0.0007  0.9957   0.0007  
        series        female      0.0645  0.0004   0.0004  
                      male        0.1764  0.0008   0.0008  
weekend documentary   female      0.0327  0.0327   0.0327  
                      male        0.0298  0.0298   0.0298  
        entertainment female      0.0015  0.0015   0.0015  
                      male        0.0014  0.0014   0.0014  
        movie         female      0.0404  0.0109   0.2588  
                      male        0.2295  0.0130   0.1508  
        news          female      0.0317  0.8095   0.0317  
                      male        0.0319  0.8088   0.0319  
        series        female      0.0919  0.0016   0.0016  
                      male        0.1840  0.0017   0.0017